<a href="https://colab.research.google.com/github/renukabhargavi2005/TEXT_MINING_CHATBOT/blob/main/TM_3_MY_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install gradio

In [ ]:
# ============================================================
# MediNLP - NLP Healthcare Chatbot
# Jupyter Notebook Version
# ============================================================

import re
import gradio as gr


# ------------------------------------------------------------
# Symptom Knowledge Base
# ------------------------------------------------------------

SYMPTOMS = {

    "fever": {
        "keywords": [
            "fever",
            "temperature",
            "high temperature"
        ],
        "conditions": [
            "Common viral infection",
            "Flu",
            "Other infections"
        ],
        "advice":
            "Rest, drink adequate fluids, and monitor your temperature."
    },

    "headache": {
        "keywords": [
            "headache",
            "head pain",
            "pain in head"
        ],
        "conditions": [
            "Tension headache",
            "Migraine",
            "Dehydration"
        ],
        "advice":
            "Rest, stay hydrated, and reduce excessive screen exposure."
    },

    "cough": {
        "keywords": [
            "cough",
            "coughing"
        ],
        "conditions": [
            "Common cold",
            "Flu",
            "Respiratory infection"
        ],
        "advice":
            "Stay hydrated and avoid smoke or other respiratory irritants."
    },

    "cold": {
        "keywords": [
            "cold",
            "runny nose",
            "blocked nose",
            "stuffy nose"
        ],
        "conditions": [
            "Common cold",
            "Allergy",
            "Viral infection"
        ],
        "advice":
            "Rest, drink fluids, and keep yourself warm."
    },

    "sore throat": {
        "keywords": [
            "sore throat",
            "throat pain",
            "painful throat"
        ],
        "conditions": [
            "Viral infection",
            "Common cold",
            "Throat infection"
        ],
        "advice":
            "Drink warm fluids and stay hydrated."
    },

    "stomach pain": {
        "keywords": [
            "stomach pain",
            "abdominal pain",
            "belly pain"
        ],
        "conditions": [
            "Indigestion",
            "Gastritis",
            "Gastrointestinal infection"
        ],
        "advice":
            "Drink fluids and avoid heavy or irritating foods."
    },

    "vomiting": {
        "keywords": [
            "vomiting",
            "vomit",
            "throwing up"
        ],
        "conditions": [
            "Gastrointestinal infection",
            "Food-related illness",
            "Indigestion"
        ],
        "advice":
            "Take small sips of water or oral rehydration solution."
    },

    "diarrhea": {
        "keywords": [
            "diarrhea",
            "loose motion",
            "loose motions"
        ],
        "conditions": [
            "Gastrointestinal infection",
            "Food-related illness"
        ],
        "advice":
            "Maintain hydration, preferably with an oral rehydration solution."
    },

    "fatigue": {
        "keywords": [
            "fatigue",
            "tired",
            "weakness",
            "weak"
        ],
        "conditions": [
            "Lack of sleep",
            "Dehydration",
            "Viral illness"
        ],
        "advice":
            "Get adequate sleep, stay hydrated, and maintain regular meals."
    }
}


# ------------------------------------------------------------
# Emergency Symptoms
# ------------------------------------------------------------

EMERGENCY_KEYWORDS = [

    "chest pain",
    "difficulty breathing",
    "cannot breathe",
    "severe breathing",
    "unconscious",
    "fainted",
    "severe bleeding",
    "stroke",
    "seizure",
    "suicidal",
    "severe allergic reaction"

]


# ------------------------------------------------------------
# NLP PREPROCESSING
# ------------------------------------------------------------

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        "",
        text
    )

    tokens = text.split()

    return text, tokens


# ------------------------------------------------------------
# SYMPTOM EXTRACTION
# ------------------------------------------------------------

def extract_symptoms(text):

    clean_text, tokens = preprocess(text)

    detected_symptoms = []

    for symptom, data in SYMPTOMS.items():

        for keyword in data["keywords"]:

            if keyword in clean_text:

                if symptom not in detected_symptoms:

                    detected_symptoms.append(symptom)

                break

    return detected_symptoms


# ------------------------------------------------------------
# EMERGENCY DETECTION
# ------------------------------------------------------------

def emergency_check(text):

    clean_text, _ = preprocess(text)

    for keyword in EMERGENCY_KEYWORDS:

        if keyword in clean_text:

            return True

    return False


# ------------------------------------------------------------
# CHATBOT
# ------------------------------------------------------------

def medinlp_chat(user_input, history):

    if not user_input:

        return "Please describe your symptoms."


    # Emergency detection
    if emergency_check(user_input):

        return (
            "⚠️ EMERGENCY WARNING\n\n"
            "The symptom you described may require immediate "
            "medical attention.\n\n"
            "Please contact your local emergency medical service "
            "or visit the nearest emergency department.\n\n"
            "MediNLP is an NLP educational assistant and cannot "
            "diagnose or treat emergencies."
        )


    # Greeting
    greetings = [
        "hi",
        "hello",
        "hey",
        "good morning",
        "good afternoon",
        "good evening"
    ]

    if user_input.lower().strip() in greetings:

        return (
            "Hello! I am MediNLP.\n\n"
            "I am an NLP-based healthcare assistant.\n\n"
            "Tell me your symptoms, for example:\n"
            "• I have fever and headache\n"
            "• I have cough and sore throat\n"
            "• I have stomach pain"
        )


    # Extract symptoms
    symptoms = extract_symptoms(user_input)


    if not symptoms:

        return (
            "I couldn't identify a recognized symptom.\n\n"
            "Please describe your symptoms more clearly.\n\n"
            "Example:\n"
            "\"I have fever, headache and cough.\""
        )


    # --------------------------------------------------------
    # Create Response
    # --------------------------------------------------------

    response = "### MediNLP Analysis\n\n"

    response += "**Detected Symptoms:**\n\n"

    for symptom in symptoms:

        response += f"- {symptom.title()}\n"


    # Conditions
    conditions = set()

    for symptom in symptoms:

        for condition in SYMPTOMS[symptom]["conditions"]:

            conditions.add(condition)


    response += "\n**Possible Categories:**\n\n"

    for condition in conditions:

        response += f"- {condition}\n"


    # Advice
    response += "\n**General Guidance:**\n\n"

    advice_added = set()

    for symptom in symptoms:

        advice = SYMPTOMS[symptom]["advice"]

        if advice not in advice_added:

            response += f"- {advice}\n"

            advice_added.add(advice)


    response += (
        "\n### Important\n\n"
        "This response is generated using an NLP-based "
        "educational system. It is not a medical diagnosis. "
        "If symptoms are severe, persistent, worsening, or "
        "concerning, consult a qualified healthcare professional."
    )


    return response


# ------------------------------------------------------------
# GRADIO INTERFACE
# ------------------------------------------------------------

demo = gr.ChatInterface(

    fn=medinlp_chat,

    title="🩺 MediNLP",

    description=(
        "Intelligent NLP Healthcare Assistant\n\n"
        "Describe your symptoms in natural language."
    ),

    examples=[
        "I have fever and headache",
        "I have cough and sore throat",
        "I have stomach pain",
        "I feel tired and weak"
    ]

)


# ------------------------------------------------------------
# Launch
# ------------------------------------------------------------

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba6e3aa44773d42e56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
